# 01 — Pulling and mapping GBIF occurrence data

This notebook pulls red squirrel (*Sciurus vulgaris*) occurrence records from GBIF for Aberdeenshire and maps them.

The central point is not the map itself but what it reveals: records cluster around roads, towns, and popular recording sites. This is **presence-only sampling bias** — the database reflects where people go, not where the species lives. Any model trained naively on this data learns observer behaviour as much as species ecology.

In [ ]:
from dotenv import load_dotenv
import os
import time
import sqlite3

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pygbif import species as gbif_species
from pygbif import occurrences


In [ ]:
load_dotenv()

GBIF_USER  = os.environ['GBIF_USER']
GBIF_PWD   = os.environ['GBIF_PWD']
GBIF_EMAIL = os.environ['GBIF_EMAIL']

print(f'Authenticated as: {GBIF_USER}')


## 1. Find the GBIF taxon key

Every taxon in GBIF has a unique integer key. We look it up by scientific name rather than hard-coding it, so the notebook remains correct if GBIF reorganises its taxonomy.

In [ ]:
match     = gbif_species.name_backbone(name='Sciurus vulgaris', rank='species')
taxon_key = match['usageKey']

print(f"Sciurus vulgaris  ->  taxon key {taxon_key}")
print(f"Status: {match['status']},  Confidence: {match['confidence']}")


## 2. Pull occurrence records for Aberdeenshire

We use a bounding box polygon covering Aberdeenshire and request only georeferenced records without known spatial issues. The API returns 300 records per call, so we paginate until `endOfRecords` is true.

In [ ]:
# Aberdeenshire bounding box (longitude latitude, WGS84)
BBOX = 'POLYGON((-3.7 56.85,-1.75 56.85,-1.75 57.75,-3.7 57.75,-3.7 56.85))'

records = []
limit   = 300
offset  = 0

while True:
    batch = occurrences.search(
        taxonKey           = taxon_key,
        geometry           = BBOX,
        hasCoordinate      = True,
        hasGeospatialIssue = False,
        limit              = limit,
        offset             = offset,
    )
    records.extend(batch['results'])
    print(f"  {len(records):>6,} / {batch['count']:,}")
    if batch['endOfRecords']:
        break
    offset += limit
    time.sleep(0.3)

print(f'\nFetch complete: {len(records):,} records.')


In [ ]:
df = pd.DataFrame(records)

keep = [
    'decimalLatitude', 'decimalLongitude',
    'year', 'month', 'basisOfRecord',
    'datasetName', 'coordinateUncertaintyInMeters',
]
keep = [c for c in keep if c in df.columns]
df   = df[keep].dropna(subset=['decimalLatitude', 'decimalLongitude']).copy()

print(df.shape)
df.head()


## 3. Save to SQLite

Raw data goes into a local SQLite database so subsequent notebooks can load it without re-querying the API. The database file is gitignored.

In [ ]:
os.makedirs('data',    exist_ok=True)
os.makedirs('figures', exist_ok=True)

DB_PATH = 'data/gbif_records.db'

with sqlite3.connect(DB_PATH) as conn:
    df.to_sql('red_squirrel', conn, if_exists='replace', index=False)

print(f'Saved {len(df):,} records to {DB_PATH}')


## 4. What do we have?

In [ ]:
print(f'Records:    {len(df):,}')
print(f"Year range: {int(df['year'].min())} to {int(df['year'].max())}")
print()
print('Basis of record:')
print(df['basisOfRecord'].value_counts().to_string())


## 5. Map the records

The left panel shows raw occurrence points. The right shows sampling density as a hexbin grid. Look for clusters — those hotspots reflect observer effort, not species abundance.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 8), facecolor='#1e1f21')

for ax in axes:
    ax.set_facecolor('#2a2b2d')
    ax.tick_params(colors='#aaa')
    for spine in ax.spines.values():
        spine.set_edgecolor('#555')
    ax.set_xlim(-3.8, -1.6)
    ax.set_ylim(56.75, 57.85)
    ax.set_xlabel('Longitude', color='#aaa')
    ax.set_ylabel('Latitude',  color='#aaa')

axes[0].scatter(
    df['decimalLongitude'], df['decimalLatitude'],
    s=4, alpha=0.5, color='#00a68a', linewidths=0,
)
axes[0].set_title('Occurrence records', color='white', pad=10)

hb = axes[1].hexbin(
    df['decimalLongitude'], df['decimalLatitude'],
    gridsize=35, cmap='YlOrRd', mincnt=1,
)
cb = fig.colorbar(hb, ax=axes[1], pad=0.02)
cb.ax.yaxis.set_tick_params(color='#aaa')
plt.setp(cb.ax.yaxis.get_ticklabels(), color='#aaa')
cb.set_label('Record count', color='#aaa')
axes[1].set_title('Sampling density', color='white', pad=10)

fig.suptitle(
    'Red squirrel (Sciurus vulgaris) — Aberdeenshire GBIF records',
    color='white', y=1.01, fontsize=13,
)
plt.tight_layout()
plt.savefig('figures/01_occurrences_map.png', dpi=150,
            bbox_inches='tight', facecolor='#1e1f21')
plt.show()


## What the map tells us

The density plot makes the bias visible. High-density cells correspond to Aberdeen city, the Deeside corridor, and sites near main roads — not because red squirrels are especially abundant there, but because these are places people visit and submit records.

This is the core problem: the training data for any SDM fitted to these records is a sample of human behaviour, not a census of the species. The next notebook addresses this through pseudo-absence generation strategies.